In [1]:
import os
print(os.getcwd())

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/MLPS'
os.makedirs(PROJECT_PATH, exist_ok=True)
os.chdir(PROJECT_PATH)
print(f"Current working directory: {os.getcwd()}")

/content
Mounted at /content/drive
Current working directory: /content/drive/MyDrive/MLPS


Version 1 (Peak Outage)

Version 1 focuses on making sure the worst hour can be covered.

It assumes that the most important goal is:

during the peak outage moment, enough households should still have backup power.

So demand is defined by:

Demand


=
t
max
y
^
it
	​

Version 2 (Sum Outage)

Version 2 focuses on the total outage burden across the full 48-hour event.

It assumes that the goal is:

reduce the overall outage impact during the whole event, not just the worst hour.

So demand is defined by:

Demand
i
	​

=
t=1
∑
48
y
^
it
	​

In [3]:
import pandas as pd
import numpy as np

pred_df = pd.read_csv("./lstm_pred_48h.csv")

pred_df["timestamp"] = pd.to_datetime(pred_df["timestamp"])
pred_df["location"] = pred_df["location"].astype(str)
pred_df["pred"] = pred_df["pred"].astype(float)

print(pred_df.head())
print(pred_df.shape)
print(pred_df["location"].nunique())

            timestamp location        pred
0 2023-06-30 01:00:00    26125  472.751770
1 2023-06-30 02:00:00    26125  504.794739
2 2023-06-30 03:00:00    26125  531.426208
3 2023-06-30 04:00:00    26125  558.316467
4 2023-06-30 05:00:00    26125  574.035278
(3984, 3)
83


In [5]:
def greedy_allocation_peak(pred_df, n_generators=5, capacity=1000):
    """
    Version 1:
    Demand_i = max hourly predicted outage over 48h
    Capacity = 1000
    """

    demand_df = (
        pred_df.groupby("location", as_index=False)["pred"]
        .max()
        .rename(columns={"pred": "demand"})
    )

    demand_df["generators_assigned"] = 0
    allocation = []

    for _ in range(n_generators):
        demand_df["marginal_benefit"] = np.minimum(
            np.maximum(
                demand_df["demand"] - capacity * demand_df["generators_assigned"],
                0), capacity
        )   # calculating marginal benefit

        # updating allocation
        best_idx = demand_df["marginal_benefit"].idxmax()
        best_county = demand_df.loc[best_idx, "location"]

        demand_df.loc[best_idx, "generators_assigned"] += 1
        allocation.append(best_county)

    demand_df["covered"] = np.minimum(
        demand_df["demand"],
        demand_df["generators_assigned"] * capacity
    )

    demand_df["remaining_unmet"] = demand_df["demand"] - demand_df["covered"]

    # summary contains the final covarage of each county
    summary = demand_df.sort_values(
        ["generators_assigned", "demand"],
        ascending=[False, False]
    ).reset_index(drop=True)

    return allocation, summary

In [6]:
def greedy_allocation_sum(pred_df, n_generators=5, hourly_capacity=1000):
    """
    Version 2:
    Demand_i = sum of hourly predicted outage over 48h
    Capacity = 1000 * number of forecast hours
    """

    horizon = pred_df["timestamp"].nunique()
    capacity = hourly_capacity * horizon

    demand_df = (
        pred_df.groupby("location", as_index=False)["pred"]
        .sum()
        .rename(columns={"pred": "demand"})
    )

    demand_df["generators_assigned"] = 0
    allocation = []

    for _ in range(n_generators):
        demand_df["marginal_benefit"] = np.minimum(
            np.maximum(
                demand_df["demand"] - capacity * demand_df["generators_assigned"],
                0
            ),
            capacity
        )

        best_idx = demand_df["marginal_benefit"].idxmax()
        best_county = demand_df.loc[best_idx, "location"]

        demand_df.loc[best_idx, "generators_assigned"] += 1
        allocation.append(best_county)

    demand_df["covered"] = np.minimum(
        demand_df["demand"],
        demand_df["generators_assigned"] * capacity
    )

    demand_df["remaining_unmet"] = demand_df["demand"] - demand_df["covered"]

    summary = demand_df.sort_values(
        ["generators_assigned", "demand"],
        ascending=[False, False]
    ).reset_index(drop=True)

    return allocation, summary

In [7]:
allocation_peak, summary_peak = greedy_allocation_peak(pred_df)

print("Version 1 Allocation: Demand = Peak Outage")
print(allocation_peak)

print(summary_peak.head(10))

Version 1 Allocation: Demand = Peak Outage
['26125', '26123', '26121', '26163', '26159']
  location      demand  generators_assigned  marginal_benefit     covered  \
0    26125  663.789673                    1          0.000000  663.789673   
1    26123  553.841858                    1          0.000000  553.841858   
2    26121  405.263550                    1          0.000000  405.263550   
3    26163  368.170624                    1          0.000000  368.170624   
4    26159  360.830627                    1        360.830627  360.830627   
5    26117  338.290192                    0        338.290192    0.000000   
6    26099  198.216583                    0        198.216583    0.000000   
7    26001  159.864426                    0        159.864426    0.000000   
8    26139  142.261002                    0        142.261002    0.000000   
9    26003  102.588295                    0        102.588295    0.000000   

   remaining_unmet  
0         0.000000  
1         0.000000  


In [8]:
allocation_sum, summary_sum = greedy_allocation_sum(pred_df)

print("Version 2 Allocation: Demand = Sum Outage, Capacity = 1000 * 48")
print(allocation_sum)

print(summary_sum.head(10))

Version 2 Allocation: Demand = Sum Outage, Capacity = 1000 * 48
['26125', '26163', '26159', '26099', '26123']
  location        demand  generators_assigned  marginal_benefit       covered  \
0    26125  20462.107489                    1          0.000000  20462.107489   
1    26163  14320.985672                    1          0.000000  14320.985672   
2    26159   8266.146035                    1          0.000000   8266.146035   
3    26099   4751.214287                    1          0.000000   4751.214287   
4    26123   4701.499662                    1       4701.499662   4701.499662   
5    26121   3574.942052                    0       3574.942052      0.000000   
6    26139   3263.921360                    0       3263.921360      0.000000   
7    26003   3073.236370                    0       3073.236370      0.000000   
8    26153   2972.314898                    0       2972.314898      0.000000   
9    26117   2886.302289                    0       2886.302289      0.000000   